In [ ]:
import pickle

In [ ]:
import requests
import time
import re
import logging
import pandas as pd
from typing import List, Dict, Tuple, Optional
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class CurationTracker:
    """Tracks dropped data at every step of the KEGG to UniProt pipeline."""
    def __init__(self):
        self.missing_ligands = []
        self.missing_orthologs = []
        self.no_target_genes = []
        self.unmapped_uniprot = []

    def print_summary(self):
        print("\n" + "="*40)
        print("📊 PIPELINE DROP SUMMARY")
        print("="*40)
        print(f"Reactions missing ligands:      {len(self.missing_ligands)}")
        print(f"Reactions missing KO groups:    {len(self.missing_orthologs)}")
        print(f"KOs missing target species:     {len(self.no_target_genes)}")
        print(f"Genes failing UniProt mapping:  {len(self.unmapped_uniprot)}")
        print("="*40 + "\n")

class KeggPathwayCurator:
    # ... [__init__ and _make_request remain exactly the same as the previous version] ...
    def __init__(self, target_species: List[str]):
        self.target_species = set(target_species)
        self.session = requests.Session()
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        self.session.mount('http://', HTTPAdapter(max_retries=retries))
        self.session.mount('https://', HTTPAdapter(max_retries=retries))
        self.base_url = "http://rest.kegg.jp"

    def _make_request(self, endpoint: str) -> Optional[str]:
        try:
            response = self.session.get(f"{self.base_url}/{endpoint}", timeout=10)
            response.raise_for_status()
            time.sleep(0.1)
            return response.text
        except requests.exceptions.RequestException as e:
            logger.error(f"Network error on endpoint {endpoint}: {e}")
            return None

    def get_pathway_reactions(self, pathway_id: str) -> List[str]:
        data = self._make_request(f"link/rn/{pathway_id}")
        if not data: return []
        return [line.split('\t')[1].replace('rn:', '') for line in data.strip().split('\n') if line]

    def get_reaction_details(self, reaction_id: str) -> Tuple[List[str], List[str]]:
        data = self._make_request(f"get/rn:{reaction_id}")
        if not data: return [], []
        compounds, orthologs = [], []
        for line in data.strip().split('\n'):
            if line.startswith('EQUATION'):
                compounds = re.findall(r'C\d{5}', line)
            elif line.startswith('ORTHOLOGY') or (line.startswith(' ') and 'K' in line):
                orthologs.extend(re.findall(r'K\d{5}', line))
        return list(set(compounds)), list(set(orthologs))

    def get_ortholog_genes(self, ko_id: str) -> List[str]:
        data = self._make_request(f"link/genes/{ko_id}")
        if not data: return []
        target_genes = []
        for line in data.strip().split('\n'):
            if not line: continue
            gene_id = line.split('\t')[1]
            if gene_id.split(':')[0] in self.target_species:
                target_genes.append(gene_id)
        return target_genes

    def convert_genes_to_uniprot(self, gene_ids: List[str]) -> Dict[str, str]:
        mapping = {}
        chunk_size = 100
        for i in range(0, len(gene_ids), chunk_size):
            chunk = gene_ids[i:i + chunk_size]
            query = "+".join(chunk)
            data = self._make_request(f"conv/uniprot/{query}")
            if data:
                for line in data.strip().split('\n'):
                    if not line: continue
                    parts = line.split('\t')
                    if len(parts) == 2:
                        mapping[parts[0]] = parts[1].replace('up:', '')
        return mapping

# ==========================================
# Execution Pipeline with Tracking
# ==========================================
def run_phase_1(pathway_ids: List[str], target_species: List[str], cache) -> Tuple[pd.DataFrame, CurationTracker]:
    curator = KeggPathwayCurator(target_species=target_species)
    tracker = CurationTracker()
    dataset_rows = cache.to_dict("records")
    
    cached_pathways = cache["Pathway"].unique()
    
    for i, pathway in enumerate(pathway_ids):
        if pathway in cached_pathways:
            logger.info(f"Skipping Pathway: {pathway}")            
        else:
            logger.info(f"Processing Pathway: {pathway}")
        reactions = curator.get_pathway_reactions(pathway)
        
        logger.info(f"Found {len(reactions)}")
        
        for rxn in reactions:
            compounds, orthologs = curator.get_reaction_details(rxn)
            
            # TRACKING: Missing Ligands or Orthologs
            if not compounds:
                tracker.missing_ligands.append(rxn)
            if not orthologs:
                tracker.missing_orthologs.append(rxn)
                
            if not orthologs or not compounds:
                continue
                
            for ko in orthologs:
                genes = curator.get_ortholog_genes(ko)
                
                # TRACKING: KO exists, but not in target species
                if not genes:
                    tracker.no_target_genes.append(ko)
                    continue
                    
                uniprot_mapping = curator.convert_genes_to_uniprot(genes)
                
                for gene in genes:
                    if gene in uniprot_mapping:
                        species = gene.split(':')[0]
                        dataset_rows.append({
                            'Pathway': pathway,
                            'Reaction': rxn,
                            'Ligands': compounds, 
                            'KO_Group': ko,
                            'Species': species,
                            'KEGG_Gene': gene,
                            'UniProt_ID': uniprot_mapping[gene]
                        })
                    else:
                        # TRACKING: Gene found, but UniProt mapping failed
                        tracker.unmapped_uniprot.append(gene)
                    
        if i % 3 == 0:
            df = pd.DataFrame(dataset_rows)
            df.to_csv("dataset_164.csv")
                    
    df = pd.DataFrame(dataset_rows)
    tracker.print_summary()
    return df, tracker


    
    # final_df, tracker_stats = run_phase_1(TARGET_PATHWAYS, TARGET_SPECIES)
    # print(f"Successfully generated {len(final_df)} entries.")

In [ ]:
TARGET_SPECIES = [
    "hsa", "mmu", "dre", "dme", "cel", "gga", "xla", "bta", "spu", "cin",
    "ath", "osa", "zma", "sly", "gmx", "vvi", "ppa", "cre", "ptr", "sbi",
    "sce", "spo", "ncr", "ani", "cne", "uma", "cal", "mgr", "yli", "kla",
    "eco", "bsu", "pae", "mtu", "sau", "sco", "ccr", "syn", "vch", "hpy",
    "mja", "hvo", "sso", "pfu", "afu", "ape", "mba", "tac", "nph", "mth"
]

TARGET_PATHWAYS_OLD = [
    "map00010", "map00020", "map00030", "map00040", "map00051", "map00052", 
    "map00053", "map00190", "map00195", "map00200", "map00240", "map00710",
    "map00061", "map00062", "map00071", "map00100", "map00120", "map00561", 
    "map00564", "map00600", "map00220", "map00250", "map00260", "map00270", 
    "map00280", "map00290", "map00300", "map00310", "map00330", "map00340", 
    "map00350", "map00360", "map00380", "map00400", "map00410", "map00670", 
    "map00730", "map00740", "map00750", "map00760", "map00770", "map00780", 
    "map00790", "map00830", "map00860", "map00510", "map00520", "map00230", 
    "map00430", "map00480"
]

In [ ]:
TARGET_PATHWAYS = list(set(TARGET_PATHWAYS) - set(TARGET_PATHWAYS_OLD))

In [ ]:
len(TARGET_PATHWAYS)

In [ ]:
# CACHE IS IMPLEMENTED FOR PATHWAY NOT species
cache = pd.read_csv("dataset.csv", index_col = 0)

In [ ]:
final_df, tracker_stats = run_phase_1(TARGET_PATHWAYS, TARGET_SPECIES, cache)

In [ ]:
final_df.shape

In [ ]:
final_df.to_csv("dataset_164.csv")

In [ ]:
with open("step_1_tracker.pkl", "wb") as f:
    pickle.dump(tracker_stats, f)

In [ ]:
final_df

In [ ]:
print(final_df.sample(3))

In [ ]:
curator = KeggPathwayCurator(target_species=["hsa"])
pathway = "map00010"

In [ ]:
curator.convert_genes_to_uniprot(["sbi:8057102"])

In [ ]:
reactions = curator.get_pathway_reactions(pathway)

In [ ]:
reactions

In [ ]:
compounds, orthologs = curator.get_reaction_details("R13199")

In [ ]:
compounds, orthologs

In [ ]:
genes = curator.get_ortholog_genes("K01810")

In [ ]:
genes

In [ ]:
curator.get_ortholog_genes("K01810")

In [ ]:
x = pd.read_csv("dataset.csv", index_col = 0)

In [ ]:
x.sample(1).to_dict()

In [ ]:
import requests
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def fetch_metabolic_pathways() -> list[str]:
    """
    Queries the KEGG API to retrieve all core metabolic pathways.
    Filters specifically for the 'map00XXX' series to guarantee granular 
    enzyme-ligand reactions while excluding global maps and diseases.
    """
    url = "http://rest.kegg.jp/list/pathway"
    logger.info(f"Querying KEGG API: {url}")
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to fetch pathways from KEGG: {e}")
        return []

    core_pathways = []
    
    for line in response.text.strip().split('\n'):
        if not line:
            continue
            
        parts = line.split('\t')
        if len(parts) == 2:
            kegg_id = parts[0].replace('path:', '')
            description = parts[1]
            
            # The map00XXX block contains the core metabolism maps
            if kegg_id.startswith('map00'):
                core_pathways.append(kegg_id)
                # Uncomment the line below if you want to see the names of what you are fetching
                # logger.info(f"Found: {kegg_id} - {description}")

    logger.info(f"Successfully extracted {len(core_pathways)} core metabolic pathways.")
    return core_pathways


TARGET_PATHWAYS = fetch_metabolic_pathways()

# Print the exact Python list format so you can copy/paste it into your Phase 1 script
print("\n# Copy and paste this into your Step 1 Script:")
print(f"TARGET_PATHWAYS = {TARGET_PATHWAYS}")

In [ ]:
TARGET_PATHWAYS = ['map00010', 'map00020', 'map00030', 'map00040', 'map00051', 'map00052', 'map00053', 'map00500', 'map00620', 'map00630', 'map00640', 'map00650', 'map00660', 'map00562', 'map00566', 'map00190', 'map00195', 'map00196', 'map00710', 'map00720', 'map00680', 'map00910', 'map00920', 'map00061', 'map00062', 'map00071', 'map00073', 'map00074', 'map00100', 'map00120', 'map00121', 'map00140', 'map00561', 'map00564', 'map00565', 'map00600', 'map00590', 'map00591', 'map00592', 'map00230', 'map00240', 'map00250', 'map00260', 'map00270', 'map00280', 'map00290', 'map00300', 'map00310', 'map00220', 'map00330', 'map00340', 'map00350', 'map00360', 'map00380', 'map00400', 'map00410', 'map00430', 'map00440', 'map00450', 'map00460', 'map00470', 'map00480', 'map00520', 'map00541', 'map00510', 'map00513', 'map00512', 'map00515', 'map00514', 'map00532', 'map00534', 'map00533', 'map00531', 'map00563', 'map00601', 'map00603', 'map00604', 'map00511', 'map00540', 'map00542', 'map00550', 'map00552', 'map00571', 'map00572', 'map00543', 'map00730', 'map00740', 'map00750', 'map00760', 'map00770', 'map00780', 'map00785', 'map00790', 'map00670', 'map00830', 'map00860', 'map00130', 'map00900', 'map00902', 'map00909', 'map00904', 'map00906', 'map00905', 'map00981', 'map00908', 'map00903', 'map00907', 'map00522', 'map00253', 'map00523', 'map00940', 'map00945', 'map00941', 'map00944', 'map00942', 'map00943', 'map00946', 'map00901', 'map00403', 'map00950', 'map00960', 'map00996', 'map00232', 'map00965', 'map00966', 'map00402', 'map00311', 'map00332', 'map00261', 'map00331', 'map00521', 'map00524', 'map00525', 'map00401', 'map00404', 'map00405', 'map00333', 'map00254', 'map00975', 'map00998', 'map00999', 'map00997', 'map00362', 'map00627', 'map00364', 'map00625', 'map00361', 'map00623', 'map00622', 'map00633', 'map00642', 'map00643', 'map00791', 'map00930', 'map00363', 'map00621', 'map00626', 'map00624', 'map00365', 'map00984', 'map00980', 'map00982', 'map00983', 'map00970']

In [ ]:
len(TARGET_PATHWAYS)

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("dataset_164.csv")

In [ ]:
df.shape

In [ ]:
df2 = pd.read_csv("dataset.csv")

In [ ]:
df2.shape

In [ ]:
df["Ligands"] = df["Ligands"].apply(eval)

In [ ]:
ligands = set()
for ligand in df["Ligands"]:
    ligands = ligands.union(ligand)

In [ ]:
len(ligands)

In [ ]:
df["UniProt_ID"].nunique()

In [ ]:
df

In [ ]:
df["Ligands"].explode().shape

In [ ]:
df_ = df.explode("Ligands")

In [ ]:
df_relevant = df_[["Ligands", "UniProt_ID"]].drop_duplicates()

In [ ]:
df_relevant

In [ ]:
x = df_relevant.groupby("Ligands")["UniProt_ID"].nunique()

In [ ]:
(x / x.sum()).round(5).tail(10)

In [ ]:
df_["UniProt_ID"].nunique()

In [ ]:
df_["Ligands"].nunique()

In [ ]:
df_.groupby("UniProt_ID")["Ligands"].count()

In [ ]:
x = df_.groupby("Ligands")["UniProt_ID"].count()

In [ ]:
x / x.sum()

In [ ]:
x = df_.groupby("Ligands")["UniProt_ID"].nunique()

In [ ]:
x / x.sum()

In [ ]:
df_[~(df_["Ligands"].isin(["C00001", "C00002", "C00003", "C00004"]))]

In [ ]:
df_.shape

In [ ]:
634056 / 746585

In [ ]:

x = df_[["Ligands", "UniProt_ID"]].drop_duplicates()

In [ ]:
x

In [ ]:
df2 = pd.read_parquet("training_data.pq")

In [ ]:
x = df2[["UniProt_ID", "Single_SMILES"]].drop_duplicates()

In [ ]:
x

In [ ]:
405377 / df2.shape[0]

In [ ]:
df_ = df.explode("Ligands")

In [ ]:
df_[["UniProt_ID", "Ligands"]].drop_duplicates().shape[0]

In [ ]:

df_["Ligands"].nunique()

In [ ]:
3430 / 385215